<a href="https://colab.research.google.com/github/xyz111131/AI-Tools-for-Statistical-Research/blob/main/FashionMNIST_MLP_W%26B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets  ## domain-specific library, include datasets
from torchvision.transforms import ToTensor

In [ ]:
pip install wandb -qqq

In [ ]:
import wandb

### Login to Weights & Biases
To use Weights & Biases, you need to log in. If you don't have an account, you can create one at [wandb.ai/login](https://wandb.ai/login).

If running in Colab, you will be prompted to enter your API key, which you can find in your [W&B settings](https://wandb.ai/settings).

In [ ]:
wandb.login()

# Download FashionMNIST data

In [ ]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(), # convert to tensor and scale to [0,1]
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

The FashionMNIST dataset is a collection of 70,000 grayscale images of fashion articles, categorized into 10 classes. It serves as a direct drop-in replacement for the original MNIST dataset for benchmarking machine learning algorithms. Each image is 28x28 pixels.

In [ ]:
training_data

In [ ]:
test_data

# Construct DataLoader

iterate over dataset, supports batching, shuffling etc.

In [ ]:
from torch.utils.data import random_split

batch_size = 64

# Split train_data (70000 samples) into validation (10000) and train (60000)
val_size = 10000
train_size = len(train_data) - val_size
val_data, train_data_split = random_split(train_data, [val_size, train_size])

# Create data loaders.
train_dataloader = DataLoader(training_data_split, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in val_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

# Explanation of the tensor shape:
# N (Batch Size): 64 images are processed in one batch.
# C (Channels): 1, indicating a grayscale image (one color channel).
# H (Height): 28 pixels.
# W (Width): 28 pixels.

# Create MLP model

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
        self.activations = {}

    def get_activation(self, name):
        def hook(model, input, output):
            self.activations[name] = output.detach()
        return hook

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)

# Register hooks for monitoring activations
model.linear_relu_stack[0].register_forward_hook(model.get_activation('layer1_fc'))
model.linear_relu_stack[2].register_forward_hook(model.get_activation('layer2_fc'))
model.linear_relu_stack[4].register_forward_hook(model.get_activation('layer3_fc'))

print(model)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in the model: {total_params}")

# Model training and testing

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [ ]:
def train(dataloader, model, loss_fn, optimizer, epoch):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss_val, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")
            wandb.log({"train/loss": loss_val, "epoch": epoch, "batch": batch})

In [ ]:
def test(dataloader, model, loss_fn, epoch):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

    # Log activations
    activations_dict = {}
    for name, tensor in model.activations.items():
        activations_dict[f"activations/{name}"] = wandb.Histogram(tensor.cpu().numpy())

    wandb.log({
        "test/loss": test_loss,
        "test/accuracy": correct,
        "epoch": epoch,
        **activations_dict
    })
    return test_loss, correct

In [ ]:
import os

wandb.init(project="fashion_mnist_experiment", name="baseline_mlp")

epochs = 50
best_loss = float('inf')

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, t)
    # Use val_dataloader for evaluation during training
    val_loss, val_acc = test(val_dataloader, model, loss_fn, t)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        artifact = wandb.Artifact('best_model', type='model')
        artifact.add_file('best_model.pth')
        wandb.log_artifact(artifact)
        print("Saved new best model based on Validation Loss!")

print("Done!")
wandb.finish()

Epoch 1
-------------------------------
loss: 0.529211  [   64/60000]
loss: 0.454029  [ 6464/60000]
loss: 0.540509  [12864/60000]
loss: 0.473353  [19264/60000]
loss: 0.314739  [25664/60000]
loss: 0.553765  [32064/60000]
loss: 0.403723  [38464/60000]
loss: 0.532567  [44864/60000]
loss: 0.310356  [51264/60000]
loss: 0.414568  [57664/60000]
Test Error: 
 Accuracy: 83.4%, Avg loss: 0.465875 

Saved new best model based on Validation Loss!
Epoch 2
-------------------------------
loss: 0.569716  [   64/60000]
loss: 0.229207  [ 6464/60000]
loss: 0.489806  [12864/60000]
loss: 0.598645  [19264/60000]
loss: 0.395295  [25664/60000]
loss: 0.757511  [32064/60000]
loss: 0.480528  [38464/60000]
loss: 0.668555  [44864/60000]
loss: 0.512702  [51264/60000]
loss: 0.482740  [57664/60000]
Test Error: 
 Accuracy: 83.3%, Avg loss: 0.465770 

Saved new best model based on Validation Loss!
Epoch 3
-------------------------------
loss: 0.515923  [   64/60000]
loss: 0.317046  [ 6464/60000]
loss: 0.418567  [12864

KeyboardInterrupt: 

### Load Model from W&B Artifact
Here is how you can download the best model artifact saved during training and load it back into PyTorch for inference.

In [ ]:
import torch
import wandb

# Initialize a new W&B run specifically for inference or evaluation
run = wandb.init(project="fashion_mnist_experiment", job_type="inference")

# Fetch the 'best_model' artifact (using the 'latest' version tag)
artifact = run.use_artifact('best_model:latest', type='model')

# Download the artifact's files to a local directory
artifact_dir = artifact.download()

# Initialize a fresh instance of the model
loaded_model = NeuralNetwork().to(device)

# Load the saved state dictionary into the model
loaded_model.load_state_dict(torch.load(f"{artifact_dir}/best_model.pth", weights_only=True))
loaded_model.eval()

print("Model loaded successfully from W&B artifact!")

# Finish the W&B run
wandb.finish()

In [ ]:


# Initialize a run for evaluation so the test() function can log to it
run = wandb.init(project="fashion_mnist_experiment", job_type="evaluation", name="eval_loaded_model")

print("Evaluating the loaded model on the test dataset...")
# Run the test loop
test_loss, test_acc = test(test_dataloader, loaded_model, loss_fn, epoch=0)

print(f"\nEvaluation complete! Accuracy: {(test_acc*100):.2f}%, Loss: {test_loss:.4f}")

# Finish the run
wandb.finish()

In [ ]:
import os

print(f"Contents of {artifact_dir}:")
print(os.listdir(artifact_dir))